# Enhanced RGB Model Training with VEML6040 Sensor (Google Colab)

This notebook trains enhanced RGB color models using VEML6040 sensor data with advanced feature engineering.

## Features:
1. Lux-based normalization using sensor sensitivity
2. White channel ratio features
3. Integration time compensation
4. Cross-channel interaction features based on spectral overlap


## 학습 목표

이 노트북을 통해 다음을 학습합니다:

1. **VEML6040 센서 이해**: RGB 색상 센서의 동작 원리
2. **특징 엔지니어링**: 원시 센서 데이터를 유용한 특징으로 변환
3. **머신러닝 모델링**: 선형 회귀를 사용한 색상 예측
4. **모델 평가**: MAE, R² 등 성능 지표 해석

### VEML6040 센서란?

- **16-bit RGB + White 센서**: 높은 정밀도의 색상 측정
- **4개 채널**: Red, Green, Blue, White
- **측정 범위**: 0-65535 (16-bit)
- **활용**: 색상 인식, 조명 제어, 환경 모니터링

### 노트북 구성

1-3: 환경 설정 및 라이브러리  
4: 데이터 업로드  
5-7: 데이터 전처리 및 특징 생성  
8-9: 데이터 분할 및 모델 학습  
10-15: 결과 분석 및 저장


## 1. Install Required Libraries


In [3]:
# Install required packages (most are pre-installed in Colab)
%pip install scikit-learn pandas numpy matplotlib seaborn -q


Note: you may need to restart the kernel to use updated packages.


## 2. Import Libraries


In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
from google.colab import files

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")


✓ All libraries imported successfully!


### 특징 엔지니어링의 중요성

**왜 원시 센서 값만으로는 부족한가?**

센서에서 직접 얻은 RAW_R, RAW_G, RAW_B, RAW_W 값은:
- 조명 밝기에 크게 영향을 받음
- 채널 간 상호작용을 반영하지 못함
- 물리적 의미(lux)와 직접 연결되지 않음

**특징 엔지니어링 기법:**

1. **Lux 정규화**: 밝기에 무관한 색상 정보 추출
2. **White 채널 비율**: 조명 조건 정규화
3. **Cross-channel 특징**: 색상 간 상대적 관계 포착

이러한 특징들은 모델이 다양한 조명 환경에서도 정확한 색상을 예측할 수 있게 합니다.


## 3. Configuration and Sensor Specifications


In [3]:
# VEML6040 Sensor Specifications
VEML6040_SPECS = {
    'peak_wavelength': {'R': 650, 'G': 550, 'B': 450},  # nm
    'bandwidth': {'R': 35, 'G': 35, 'B': 40},  # nm (±)
    'sensitivity': 0.007865,  # lux/step
    'max_count': 65535,  # 16-bit
    'integration_time_default': 40,  # ms (typical, may vary)
}

# Configuration
TEST_SIZE = 0.2
RANDOM_STATE = 42
POLY_DEGREE = 2

# Feature engineering options
USE_LUX_NORMALIZATION = True
USE_WHITE_RATIO = True
USE_CROSS_CHANNEL = True

print("VEML6040 Sensor Specifications:")
print(f"  Peak wavelengths: R={VEML6040_SPECS['peak_wavelength']['R']}nm, "
      f"G={VEML6040_SPECS['peak_wavelength']['G']}nm, "
      f"B={VEML6040_SPECS['peak_wavelength']['B']}nm")
print(f"  Sensitivity: {VEML6040_SPECS['sensitivity']} lux/step")
print(f"  Resolution: 16-bit (0-{VEML6040_SPECS['max_count']})")
print(f"\nFeature Engineering Options:")
print(f"  Lux normalization: {USE_LUX_NORMALIZATION}")
print(f"  White channel ratios: {USE_WHITE_RATIO}")
print(f"  Cross-channel features: {USE_CROSS_CHANNEL}")


VEML6040 Sensor Specifications:
  Peak wavelengths: R=650nm, G=550nm, B=450nm
  Sensitivity: 0.007865 lux/step
  Resolution: 16-bit (0-65535)

Feature Engineering Options:
  Lux normalization: True
  White channel ratios: True
  Cross-channel features: True


## 4. Upload Training Data CSV

이 단계에서는 학습에 사용할 CSV 데이터를 업로드합니다.

**CSV 파일 구조:**
- `NO`: 샘플 번호
- `RAW_R`, `RAW_G`, `RAW_B`, `RAW_W`: VEML6040 센서의 원시(raw) 측정값
- `R_255`, `G_255`, `B_255`: 목표 RGB 값 (0-255 범위)

**왜 필요한가?**
- 실제 센서 데이터로부터 RGB 색상 예측 모델을 학습하기 위해
- 센서 값과 실제 색상 간의 관계를 파악하기 위해


In [ ]:
# Upload CSV file
print("Please upload your CSV file...")
uploaded = files.upload()

# Get the uploaded filename
CSV_FILE = list(uploaded.keys())[0]
print(f"\n✓ Uploaded file: {CSV_FILE}")


Please upload your CSV file...


NameError: name 'load_and_preprocess_data' is not defined

### 특징 계산 함수 이해하기

이 섹션에서는 특징 엔지니어링에 필요한 함수들을 정의합니다.

#### 1. `calculate_lux()` - Lux 값 계산
- VEML6040 센서의 감도(0.007865 lux/step)를 사용
- White 채널 값을 실제 조명 밝기(lux)로 변환
- 공식: `Lux = RAW_W × 0.007865`

#### 2. `calculate_white_ratios()` - White 채널 비율
- 각 색상 채널을 White 채널로 나눔
- 조명 밝기의 영향을 정규화
- 0으로 나누기 방지를 위한 안전 장치 포함

#### 3. `calculate_cross_channel_features()` - 채널 간 특징
- **Color Dominance**: 전체 색상에서 각 채널이 차지하는 비율
- **Color Ratios**: R/G, G/B, R/B 비율로 색상 관계 표현

#### 4. `engineer_features()` - 통합 특징 생성
- 위의 모든 함수를 조합하여 최종 특징 DataFrame 생성
- 설정에 따라 선택적으로 특징 추가 가능


## 5. Feature Engineering Functions


In [ ]:
def calculate_lux(raw_w):
    """
    Calculate illuminance in lux from raw white channel
    Based on VEML6040 sensitivity: 0.007865 lux/step
    """
    return raw_w * VEML6040_SPECS['sensitivity']


def calculate_white_ratios(raw_r, raw_g, raw_b, raw_w):
    """
    Calculate ratio of each color channel to white channel
    This helps normalize for different lighting conditions
    """
    # Avoid division by zero
    raw_w_safe = np.maximum(raw_w, 1)
    
    ratio_r = raw_r / raw_w_safe
    ratio_g = raw_g / raw_w_safe
    ratio_b = raw_b / raw_w_safe
    
    return ratio_r, ratio_g, ratio_b


def calculate_cross_channel_features(raw_r, raw_g, raw_b):
    """
    Calculate cross-channel features based on spectral overlap
    VEML6040 channels have overlapping spectral responses
    """
    # Color dominance
    total = raw_r + raw_g + raw_b + 1e-6  # avoid division by zero
    dom_r = raw_r / total
    dom_g = raw_g / total
    dom_b = raw_b / total
    
    # Color ratios
    rg_ratio = raw_r / (raw_g + 1)
    gb_ratio = raw_g / (raw_b + 1)
    rb_ratio = raw_r / (raw_b + 1)
    
    return dom_r, dom_g, dom_b, rg_ratio, gb_ratio, rb_ratio


def engineer_features(df):
    """
    Create engineered features based on VEML6040 characteristics
    
    Returns:
        Enhanced feature dataframe
    """
    features = {}
    
    # Original raw values
    features['RAW_R'] = df['RAW_R'].values
    features['RAW_G'] = df['RAW_G'].values
    features['RAW_B'] = df['RAW_B'].values
    features['RAW_W'] = df['RAW_W'].values
    
    if USE_LUX_NORMALIZATION:
        # Lux-based normalization
        lux = calculate_lux(df['RAW_W'].values)
        features['LUX'] = lux
        
        # Normalized by lux (brightness-independent color)
        features['R_NORM_LUX'] = df['RAW_R'].values / (lux + 1)
        features['G_NORM_LUX'] = df['RAW_G'].values / (lux + 1)
        features['B_NORM_LUX'] = df['RAW_B'].values / (lux + 1)
    
    if USE_WHITE_RATIO:
        # White channel ratios
        ratio_r, ratio_g, ratio_b = calculate_white_ratios(
            df['RAW_R'].values, 
            df['RAW_G'].values, 
            df['RAW_B'].values, 
            df['RAW_W'].values
        )
        features['RATIO_R_W'] = ratio_r
        features['RATIO_G_W'] = ratio_g
        features['RATIO_B_W'] = ratio_b
    
    if USE_CROSS_CHANNEL:
        # Cross-channel features
        dom_r, dom_g, dom_b, rg_ratio, gb_ratio, rb_ratio = calculate_cross_channel_features(
            df['RAW_R'].values,
            df['RAW_G'].values,
            df['RAW_B'].values
        )
        features['DOM_R'] = dom_r
        features['DOM_G'] = dom_g
        features['DOM_B'] = dom_b
        features['RG_RATIO'] = rg_ratio
        features['GB_RATIO'] = gb_ratio
        features['RB_RATIO'] = rb_ratio
    
    return pd.DataFrame(features)

print("✓ Feature engineering functions defined!")


## 6. Load and Preprocess Data

이제 업로드한 CSV 파일을 로드하고 전처리를 시작합니다.

**전처리 과정:**
1. **CSV 파일 로드**: 데이터를 메모리로 읽어들임
2. **데이터 정제**: 공백 제거, 타입 변환, 결측치 처리
3. **기본 특징 추출**: RAW 센서 값 추출
4. **고급 특징 생성**: Lux 정규화, White 채널 비율, Cross-channel 특징

**주목할 점:**
- 각 단계의 결과를 확인하면서 진행
- 데이터의 품질과 분포를 이해
- 특징 엔지니어링이 모델 성능에 미치는 영향 관찰


In [ ]:
# Step 1: CSV 파일 로드
print("=" * 60)
print("Step 1: CSV 파일 로드")
print("=" * 60)

df = pd.read_csv(CSV_FILE)

print(f"✓ CSV 파일 로드 완료!")
print(f"  총 샘플 수: {len(df)}")
print(f"  컬럼 목록: {list(df.columns)}")
print(f"\n처음 5개 샘플:")
display(df.head())


In [ ]:
# Step 2: 데이터 정제 (공백 제거, 타입 변환, 결측치 처리)
print("\n" + "=" * 60)
print("Step 2: 데이터 정제")
print("=" * 60)

# Remove spaces from column values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()

# Convert to numeric
numeric_columns = ['NO', 'RAW_R', 'RAW_G', 'RAW_B', 'RAW_W', 'R_255', 'G_255', 'B_255']
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Check for missing values before dropping
missing_before = df.isnull().sum().sum()
print(f"  결측치 개수 (처리 전): {missing_before}")

# Remove rows with missing values
df = df.dropna()

print(f"✓ 데이터 정제 완료!")
print(f"  정제 후 샘플 수: {len(df)}")
print(f"\n데이터 통계 요약:")
display(df[['RAW_R', 'RAW_G', 'RAW_B', 'RAW_W', 'R_255', 'G_255', 'B_255']].describe())


In [ ]:
# Step 3: 기본 특징 추출 (RAW 센서 값)
print("\n" + "=" * 60)
print("Step 3: 기본 RAW 특징 추출")
print("=" * 60)

# Extract base features
features = {}
features['RAW_R'] = df['RAW_R'].values
features['RAW_G'] = df['RAW_G'].values
features['RAW_B'] = df['RAW_B'].values
features['RAW_W'] = df['RAW_W'].values

print("✓ 기본 RAW 특징 추출 완료")
print(f"  RAW_R 범위: {features['RAW_R'].min():.0f} ~ {features['RAW_R'].max():.0f}")
print(f"  RAW_G 범위: {features['RAW_G'].min():.0f} ~ {features['RAW_G'].max():.0f}")
print(f"  RAW_B 범위: {features['RAW_B'].min():.0f} ~ {features['RAW_B'].max():.0f}")
print(f"  RAW_W 범위: {features['RAW_W'].min():.0f} ~ {features['RAW_W'].max():.0f}")

print(f"\n💡 센서 값은 16-bit (0-65535) 범위입니다.")


In [ ]:
# Step 4: Lux 정규화 특징 추가
print("\n" + "=" * 60)
print("Step 4: Lux 정규화 특징 생성")
print("=" * 60)

if USE_LUX_NORMALIZATION:
    # Calculate lux from white channel
    lux = calculate_lux(df['RAW_W'].values)
    features['LUX'] = lux
    
    # Normalize by lux (brightness-independent color)
    features['R_NORM_LUX'] = df['RAW_R'].values / (lux + 1)
    features['G_NORM_LUX'] = df['RAW_G'].values / (lux + 1)
    features['B_NORM_LUX'] = df['RAW_B'].values / (lux + 1)
    
    print("✓ Lux 정규화 특징 추가 완료")
    print(f"  Lux 범위: {lux.min():.2f} ~ {lux.max():.2f} lux")
    print(f"  R/Lux 범위: {features['R_NORM_LUX'].min():.2f} ~ {features['R_NORM_LUX'].max():.2f}")
    print(f"  G/Lux 범위: {features['G_NORM_LUX'].min():.2f} ~ {features['G_NORM_LUX'].max():.2f}")
    print(f"  B/Lux 범위: {features['B_NORM_LUX'].min():.2f} ~ {features['B_NORM_LUX'].max():.2f}")
    print(f"\n💡 Lux 정규화는 조명 밝기에 무관한 색상 정보를 추출합니다.")
else:
    print("⊗ Lux 정규화 비활성화됨")


### Lux 분포 시각화

Lux 값의 분포를 확인하여 데이터의 밝기 범위를 이해합니다.


In [ ]:
# Visualize Lux distribution
if USE_LUX_NORMALIZATION:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Lux histogram
    axes[0].hist(features['LUX'], bins=30, color='orange', alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('Lux Value', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title('Lux Distribution', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Lux vs Raw_W scatter
    axes[1].scatter(df['RAW_W'], features['LUX'], alpha=0.6, color='orange', s=30, edgecolors='black', linewidth=0.5)
    axes[1].set_xlabel('RAW_W (White Channel)', fontsize=12)
    axes[1].set_ylabel('Lux', fontsize=12)
    axes[1].set_title('RAW_W vs Lux (Linear Relationship)', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Step 5: White 채널 비율 특징 추가
print("\n" + "=" * 60)
print("Step 5: White 채널 비율 특징 생성")
print("=" * 60)

if USE_WHITE_RATIO:
    # White channel ratios
    ratio_r, ratio_g, ratio_b = calculate_white_ratios(
        df['RAW_R'].values, 
        df['RAW_G'].values, 
        df['RAW_B'].values, 
        df['RAW_W'].values
    )
    features['RATIO_R_W'] = ratio_r
    features['RATIO_G_W'] = ratio_g
    features['RATIO_B_W'] = ratio_b
    
    print("✓ White 채널 비율 특징 추가 완료")
    print(f"  R/W 비율 범위: {ratio_r.min():.3f} ~ {ratio_r.max():.3f}")
    print(f"  G/W 비율 범위: {ratio_g.min():.3f} ~ {ratio_g.max():.3f}")
    print(f"  B/W 비율 범위: {ratio_b.min():.3f} ~ {ratio_b.max():.3f}")
    print(f"\n💡 White 채널 비율은 다양한 조명 조건을 정규화하는데 도움이 됩니다.")
else:
    print("⊗ White 채널 비율 비활성화됨")


### White 채널 비율 시각화

각 색상 채널이 White 채널에 대해 어떤 비율을 가지는지 확인합니다.


In [ ]:
# Visualize White channel ratios
if USE_WHITE_RATIO:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    colors = ['red', 'green', 'blue']
    ratio_names = ['RATIO_R_W', 'RATIO_G_W', 'RATIO_B_W']
    titles = ['Red/White Ratio', 'Green/White Ratio', 'Blue/White Ratio']
    
    for idx, (color, ratio_name, title) in enumerate(zip(colors, ratio_names, titles)):
        axes[idx].scatter(df['RAW_W'], features[ratio_name], 
                         alpha=0.5, color=color, s=20, edgecolors='black', linewidth=0.3)
        axes[idx].set_xlabel('RAW_W (White Channel)', fontsize=11)
        axes[idx].set_ylabel(f'{ratio_name}', fontsize=11)
        axes[idx].set_title(title, fontsize=13, fontweight='bold')
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Step 6: Cross-channel 특징 추가
print("\n" + "=" * 60)
print("Step 6: Cross-channel 특징 생성")
print("=" * 60)

if USE_CROSS_CHANNEL:
    # Cross-channel features
    dom_r, dom_g, dom_b, rg_ratio, gb_ratio, rb_ratio = calculate_cross_channel_features(
        df['RAW_R'].values,
        df['RAW_G'].values,
        df['RAW_B'].values
    )
    features['DOM_R'] = dom_r
    features['DOM_G'] = dom_g
    features['DOM_B'] = dom_b
    features['RG_RATIO'] = rg_ratio
    features['GB_RATIO'] = gb_ratio
    features['RB_RATIO'] = rb_ratio
    
    print("✓ Cross-channel 특징 추가 완료")
    print(f"  Color Dominance:")
    print(f"    - R 지배도 범위: {dom_r.min():.3f} ~ {dom_r.max():.3f}")
    print(f"    - G 지배도 범위: {dom_g.min():.3f} ~ {dom_g.max():.3f}")
    print(f"    - B 지배도 범위: {dom_b.min():.3f} ~ {dom_b.max():.3f}")
    print(f"  Color Ratios:")
    print(f"    - R/G 비율 범위: {rg_ratio.min():.3f} ~ {rg_ratio.max():.3f}")
    print(f"    - G/B 비율 범위: {gb_ratio.min():.3f} ~ {gb_ratio.max():.3f}")
    print(f"    - R/B 비율 범위: {rb_ratio.min():.3f} ~ {rb_ratio.max():.3f}")
    print(f"\n💡 Cross-channel 특징은 채널 간 상호작용과 색상의 상대적 강도를 나타냅니다.")
else:
    print("⊗ Cross-channel 특징 비활성화됨")


### Cross-channel 특징 상관관계

Color Dominance 특징들 간의 관계를 확인합니다. 세 채널의 지배도 합은 항상 1입니다.


In [ ]:
# Visualize Cross-channel features
if USE_CROSS_CHANNEL:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # DOM_R vs DOM_G
    axes[0].scatter(features['DOM_R'], features['DOM_G'], 
                   alpha=0.5, c=df['R_255'], cmap='RdYlGn', s=30, edgecolors='black', linewidth=0.3)
    axes[0].set_xlabel('Red Dominance', fontsize=11)
    axes[0].set_ylabel('Green Dominance', fontsize=11)
    axes[0].set_title('Red vs Green Dominance', fontsize=13, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # DOM_G vs DOM_B
    axes[1].scatter(features['DOM_G'], features['DOM_B'], 
                   alpha=0.5, c=df['G_255'], cmap='summer', s=30, edgecolors='black', linewidth=0.3)
    axes[1].set_xlabel('Green Dominance', fontsize=11)
    axes[1].set_ylabel('Blue Dominance', fontsize=11)
    axes[1].set_title('Green vs Blue Dominance', fontsize=13, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    # Verify sum = 1
    dom_sum = features['DOM_R'] + features['DOM_G'] + features['DOM_B']
    axes[2].hist(dom_sum, bins=30, color='purple', alpha=0.7, edgecolor='black')
    axes[2].set_xlabel('DOM_R + DOM_G + DOM_B', fontsize=11)
    axes[2].set_ylabel('Frequency', fontsize=11)
    axes[2].set_title('Dominance Sum (Should be 1.0)', fontsize=13, fontweight='bold')
    axes[2].axvline(x=1.0, color='red', linestyle='--', linewidth=2, label='Expected = 1.0')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Step 7: 최종 특징 DataFrame 생성 및 타겟 변수 설정
print("\n" + "=" * 60)
print("Step 7: 최종 특징 데이터 준비")
print("=" * 60)

# Create feature DataFrame
X_df = pd.DataFrame(features)

# Target values
y_r = df['R_255'].values
y_g = df['G_255'].values
y_b = df['B_255'].values

print("✓ 최종 특징 생성 완료!")
print(f"\n{'='*60}")
print("특징 생성 요약")
print(f"{'='*60}")
print(f"  총 특징 개수: {X_df.shape[1]}")
print(f"  샘플 수: {len(X_df)}")
print(f"  특징 이름 목록:")
for i, name in enumerate(X_df.columns, 1):
    print(f"    {i:2d}. {name}")

print(f"\n타겟 변수:")
print(f"  R_255 범위: {y_r.min():.0f} ~ {y_r.max():.0f}")
print(f"  G_255 범위: {y_g.min():.0f} ~ {y_g.max():.0f}")
print(f"  B_255 범위: {y_b.min():.0f} ~ {y_b.max():.0f}")

print(f"\n처음 5개 샘플의 특징값:")
display(X_df.head())

# Prepare for later use
X = X_df.values
feature_names = X_df.columns.tolist()

print(f"\n✓ 데이터 전처리 완료! 이제 학습할 준비가 되었습니다.")


### Train/Test 데이터 분할의 중요성

**왜 데이터를 나누는가?**

머신러닝 모델의 목표는 **일반화(generalization)**입니다:
- 학습 데이터에만 잘 맞는 것이 아니라
- **처음 보는 새로운 데이터**에도 정확한 예측을 해야 합니다

**Train/Test Split:**
- **Train Set (80%)**: 모델 학습에 사용
- **Test Set (20%)**: 모델 성능 평가에 사용
- Test Set은 학습 과정에서 전혀 사용되지 않음

**과적합(Overfitting) 방지:**
- Train 성능은 좋지만 Test 성능이 나쁘다면? → 과적합!
- Train과 Test 성능이 모두 좋아야 → 좋은 모델

**RANDOM_STATE = 42:**
- 재현 가능한 결과를 위한 랜덤 시드
- 같은 코드를 다시 실행해도 동일한 분할 결과


## 7. Visualize Data Distribution

데이터의 분포를 시각화하여 이해합니다.

**시각화 내용:**
- **원시 센서 값 분포**: RAW_R, RAW_G, RAW_B, RAW_W의 히스토그램
- **타겟 RGB 값 분포**: 목표로 하는 0-255 범위의 RGB 값 분포

**왜 중요한가?**
- 데이터의 범위와 분포를 확인
- 이상치(outlier) 존재 여부 파악
- 데이터 불균형 확인


In [ ]:
# Plot raw sensor readings distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Raw Sensor Readings Distribution', fontsize=16, fontweight='bold')

for idx, (channel, color) in enumerate([('RAW_R', 'red'), ('RAW_G', 'green'), 
                                         ('RAW_B', 'blue'), ('RAW_W', 'gray')]):
    ax = axes[idx // 2, idx % 2]
    ax.hist(df[channel], bins=30, color=color, alpha=0.7, edgecolor='black')
    ax.set_xlabel(f'{channel} Value', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title(f'{channel} Distribution', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Plot target RGB distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Target RGB Values (0-255) Distribution', fontsize=16, fontweight='bold')

for idx, (channel, color, y_data) in enumerate([('R', 'red', y_r), 
                                                  ('G', 'green', y_g), 
                                                  ('B', 'blue', y_b)]):
    axes[idx].hist(y_data, bins=30, color=color, alpha=0.7, edgecolor='black')
    axes[idx].set_xlabel(f'{channel}_255 Value', fontsize=12)
    axes[idx].set_ylabel('Frequency', fontsize=12)
    axes[idx].set_title(f'{channel} Channel', fontsize=14, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 모델 학습: Linear Regression vs Ridge Regression

**선형 회귀(Linear Regression)란?**

특징(features)과 타겟(target) 간의 선형 관계를 학습:
```
RGB_predicted = w1*feature1 + w2*feature2 + ... + wn*featuren + b
```

**왜 R, G, B를 따로 학습하는가?**
- 각 색상 채널은 독립적인 패턴을 가짐
- 채널별로 최적의 가중치(weights)가 다름
- 3개의 독립적인 모델이 더 정확

**Linear Regression vs Ridge Regression:**

| 특징 | Linear Regression | Ridge Regression |
|------|-------------------|------------------|
| 정규화 | 없음 | L2 정규화 (alpha=1.0) |
| 과적합 방지 | 약함 | 강함 |
| 특징 많을 때 | 과적합 위험 | 안정적 |
| 계산 복잡도 | 낮음 | 약간 높음 |

**자동 모델 선택:**
- 두 모델을 모두 학습
- Test MAE가 낮은 모델 자동 선택
- 최적의 성능 보장


## 8. Split Data into Train and Test Sets


In [ ]:
print("=" * 60)
print(f"Splitting data (Train: {int((1-TEST_SIZE)*100)}%, Test: {int(TEST_SIZE*100)}%)")
print("=" * 60)

X_train, X_test, y_r_train, y_r_test = train_test_split(X, y_r, test_size=TEST_SIZE, random_state=RANDOM_STATE)
_, _, y_g_train, y_g_test = train_test_split(X, y_g, test_size=TEST_SIZE, random_state=RANDOM_STATE)
_, _, y_b_train, y_b_test = train_test_split(X, y_b, test_size=TEST_SIZE, random_state=RANDOM_STATE)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Feature dimensions: {X_train.shape[1]}")


## 9. Train Enhanced Models


In [ ]:
def train_enhanced_model(X_train, y_train, X_test, y_test, channel_name, feature_names):
    """
    Train enhanced model with regularization
    """
    # Try both standard Linear Regression and Ridge (with L2 regularization)
    model_lr = LinearRegression()
    model_lr.fit(X_train, y_train)
    
    model_ridge = Ridge(alpha=1.0)
    model_ridge.fit(X_train, y_train)
    
    # Evaluate both
    y_test_pred_lr = np.clip(model_lr.predict(X_test), 0, 255)
    y_test_pred_ridge = np.clip(model_ridge.predict(X_test), 0, 255)
    
    mae_lr = mean_absolute_error(y_test, y_test_pred_lr)
    mae_ridge = mean_absolute_error(y_test, y_test_pred_ridge)
    
    # Choose better model
    if mae_ridge < mae_lr:
        model = model_ridge
        y_test_pred = y_test_pred_ridge
        model_type = "Ridge"
    else:
        model = model_lr
        y_test_pred = y_test_pred_lr
        model_type = "Linear"
    
    y_train_pred = np.clip(model.predict(X_train), 0, 255)
    
    # Calculate metrics
    metrics = {
        'model_type': model_type,
        'train_mae': mean_absolute_error(y_train, y_train_pred),
        'test_mae': mean_absolute_error(y_test, y_test_pred),
        'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'train_r2': r2_score(y_train, y_train_pred),
        'test_r2': r2_score(y_test, y_test_pred),
        'y_test': y_test,
        'y_test_pred': y_test_pred,
        'y_train': y_train,
        'y_train_pred': y_train_pred,
        'feature_importance': dict(zip(feature_names, np.abs(model.coef_)))
    }
    
    return model, metrics


def print_feature_importance(metrics, channel_name):
    """Print top 5 most important features"""
    importance = metrics['feature_importance']
    sorted_features = sorted(importance.items(), key=lambda x: -x[1])
    
    print(f"\n  Top 5 features for {channel_name}:")
    for i, (feat, imp) in enumerate(sorted_features[:5], 1):
        print(f"    {i}. {feat:15s}: {imp:8.4f}")


# Train models for each channel
all_models = {}
all_metrics = {'r': {}, 'g': {}, 'b': {}}

for channel, y_train, y_test in [('r', y_r_train, y_r_test), 
                                  ('g', y_g_train, y_g_test), 
                                  ('b', y_b_train, y_b_test)]:
    print(f"\n{'=' * 60}")
    print(f"Training enhanced model for {channel.upper()} channel")
    print(f"{'=' * 60}")
    
    model, metrics = train_enhanced_model(X_train, y_train, X_test, y_test, channel, feature_names)
    all_models[channel] = model
    all_metrics[channel] = metrics
    
    print(f"\n{channel.upper()} Channel Results ({metrics['model_type']} Regression):")
    print(f"  Train MAE: {metrics['train_mae']:.2f}, Test MAE: {metrics['test_mae']:.2f}")
    print(f"  Train R²: {metrics['train_r2']:.4f}, Test R²: {metrics['test_r2']:.4f}")
    
    print_feature_importance(metrics, channel.upper())

print(f"\n{'=' * 60}")
print("✓ All models trained successfully!")
print(f"{'=' * 60}")


## 10. Performance Summary

모델 학습이 완료되었습니다! 이제 각 채널(R, G, B)의 성능을 종합적으로 비교합니다.

**성능 지표:**
- **MAE (Mean Absolute Error)**: 평균 절대 오차 - 낮을수록 좋음
- **R² Score**: 결정계수 (0~1) - 1에 가까울수록 좋음
- **Model Type**: 선택된 모델 유형 (Linear vs Ridge)

**해석 방법:**
- MAE < 10: 매우 좋은 성능
- MAE 10-20: 좋은 성능
- MAE > 20: 개선 필요
- R² > 0.95: 우수한 설명력


In [ ]:
print("=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)
print(f"\nEnhanced Model Performance:")
print(f"{'Channel':<10} {'Test MAE':<12} {'Test R²':<10} {'Model Type':<15}")
print("-" * 60)
for channel in ['r', 'g', 'b']:
    m = all_metrics[channel]
    print(f"{channel.upper():<10} {m['test_mae']:>11.2f} {m['test_r2']:>9.4f} {m['model_type']:<15}")

avg_mae = np.mean([all_metrics[ch]['test_mae'] for ch in ['r', 'g', 'b']])
avg_r2 = np.mean([all_metrics[ch]['test_r2'] for ch in ['r', 'g', 'b']])
print(f"{'Average':<10} {avg_mae:>11.2f} {avg_r2:>9.4f}")
print("=" * 60)


## 11. Visualize Model Performance


In [ ]:
# Plot prediction vs actual for each channel
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Predictions vs Actual Values', fontsize=16, fontweight='bold')

for idx, (channel, color) in enumerate([('r', 'red'), ('g', 'green'), ('b', 'blue')]):
    metrics = all_metrics[channel]
    ax = axes[idx]
    
    # Scatter plot
    ax.scatter(metrics['y_test'], metrics['y_test_pred'], 
               alpha=0.6, color=color, s=50, edgecolors='black', linewidth=0.5)
    
    # Perfect prediction line
    ax.plot([0, 255], [0, 255], 'k--', linewidth=2, label='Perfect Prediction')
    
    ax.set_xlabel('Actual Value', fontsize=12)
    ax.set_ylabel('Predicted Value', fontsize=12)
    ax.set_title(f'{channel.upper()} Channel\n'
                 f'MAE: {metrics["test_mae"]:.2f}, R²: {metrics["test_r2"]:.4f}', 
                 fontsize=13, fontweight='bold')
    ax.set_xlim(0, 255)
    ax.set_ylim(0, 255)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Plot residuals
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Residual Analysis (Prediction Error)', fontsize=16, fontweight='bold')

for idx, (channel, color) in enumerate([('r', 'red'), ('g', 'green'), ('b', 'blue')]):
    metrics = all_metrics[channel]
    ax = axes[idx]
    
    residuals = metrics['y_test'] - metrics['y_test_pred']
    
    ax.scatter(metrics['y_test_pred'], residuals, 
               alpha=0.6, color=color, s=50, edgecolors='black', linewidth=0.5)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=2)
    
    ax.set_xlabel('Predicted Value', fontsize=12)
    ax.set_ylabel('Residual (Actual - Predicted)', fontsize=12)
    ax.set_title(f'{channel.upper()} Channel Residuals', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 12. Feature Importance Visualization


In [ ]:
# Plot feature importance for each channel
fig, axes = plt.subplots(3, 1, figsize=(12, 12))
fig.suptitle('Feature Importance by Channel', fontsize=16, fontweight='bold')

for idx, (channel, color) in enumerate([('r', 'red'), ('g', 'green'), ('b', 'blue')]):
    importance = all_metrics[channel]['feature_importance']
    sorted_features = sorted(importance.items(), key=lambda x: -x[1])
    
    # Top 10 features
    top_features = sorted_features[:10]
    names = [f[0] for f in top_features]
    values = [f[1] for f in top_features]
    
    axes[idx].barh(names, values, color=color, alpha=0.7, edgecolor='black')
    axes[idx].set_xlabel('Absolute Coefficient Value', fontsize=12)
    axes[idx].set_title(f'{channel.upper()} Channel - Top 10 Features', 
                        fontsize=13, fontweight='bold')
    axes[idx].invert_yaxis()
    axes[idx].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


## 13. Save Models and Export Results


In [ ]:
print("=" * 60)
print("Saving enhanced models...")
print("=" * 60)

# Prepare output data
output_data = {
    'models': all_models,
    'feature_names': feature_names,
    'sensor_specs': VEML6040_SPECS,
    'feature_options': {
        'lux_normalization': USE_LUX_NORMALIZATION,
        'white_ratio': USE_WHITE_RATIO,
        'cross_channel': USE_CROSS_CHANNEL
    }
}

# Save models as pickle
pkl_filename = "rgb_models_enhanced.pkl"
with open(pkl_filename, 'wb') as f:
    pickle.dump(output_data, f)
print(f"✓ Enhanced models saved to: {pkl_filename}")

# Save feature importance as JSON
importance_data = {}
for channel in ['r', 'g', 'b']:
    importance_data[channel] = all_metrics[channel]['feature_importance']

json_filename = "feature_importance.json"
with open(json_filename, 'w') as f:
    # Convert numpy types to native Python types for JSON serialization
    importance_json = {}
    for channel, features in importance_data.items():
        importance_json[channel] = {k: float(v) for k, v in features.items()}
    json.dump(importance_json, f, indent=4)
print(f"✓ Feature importance saved to: {json_filename}")

# Save performance metrics as JSON
metrics_filename = "model_performance.json"
performance_data = {}
for channel in ['r', 'g', 'b']:
    performance_data[channel] = {
        'model_type': all_metrics[channel]['model_type'],
        'train_mae': float(all_metrics[channel]['train_mae']),
        'test_mae': float(all_metrics[channel]['test_mae']),
        'train_r2': float(all_metrics[channel]['train_r2']),
        'test_r2': float(all_metrics[channel]['test_r2'])
    }

with open(metrics_filename, 'w') as f:
    json.dump(performance_data, f, indent=4)
print(f"✓ Performance metrics saved to: {metrics_filename}")

print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)
print("\nGenerated files:")
print(f"  - {pkl_filename} (Enhanced models with metadata)")
print(f"  - {json_filename} (Feature importance analysis)")
print(f"  - {metrics_filename} (Performance metrics)")
print("=" * 60)


## 14. Download Generated Files


In [ ]:
# Download all generated files
print("Downloading files...\n")

files.download('rgb_models_enhanced.pkl')
files.download('feature_importance.json')
files.download('model_performance.json')

print("\n✓ All files downloaded successfully!")
print("\nYou can now use these models for real-time prediction.")


## 15. Test Model with Sample Data (Optional)
+


In [ ]:
# Test with a random sample from test set
sample_idx = np.random.randint(0, len(X_test))
sample_features = X_test[sample_idx:sample_idx+1]

print("Testing model with a random sample:")
print("=" * 60)
print("\nInput features (first 5):")
for i, name in enumerate(feature_names[:5]):
    print(f"  {name}: {sample_features[0, i]:.4f}")

# Predict
pred_r = np.clip(all_models['r'].predict(sample_features)[0], 0, 255)
pred_g = np.clip(all_models['g'].predict(sample_features)[0], 0, 255)
pred_b = np.clip(all_models['b'].predict(sample_features)[0], 0, 255)

# Actual values
actual_r = y_r_test[sample_idx]
actual_g = y_g_test[sample_idx]
actual_b = y_b_test[sample_idx]

print("\nPredictions:")
print(f"  R: {pred_r:.1f} (Actual: {actual_r:.1f}, Error: {abs(pred_r-actual_r):.1f})")
print(f"  G: {pred_g:.1f} (Actual: {actual_g:.1f}, Error: {abs(pred_g-actual_g):.1f})")
print(f"  B: {pred_b:.1f} (Actual: {actual_b:.1f}, Error: {abs(pred_b-actual_b):.1f})")

# Visualize colors
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Color Comparison', fontsize=14, fontweight='bold')

# Predicted color
pred_color = np.array([[[pred_r/255, pred_g/255, pred_b/255]]])
axes[0].imshow(pred_color)
axes[0].set_title(f'Predicted\nRGB({pred_r:.0f}, {pred_g:.0f}, {pred_b:.0f})', fontsize=12)
axes[0].axis('off')

# Actual color
actual_color = np.array([[[actual_r/255, actual_g/255, actual_b/255]]])
axes[1].imshow(actual_color)
axes[1].set_title(f'Actual\nRGB({actual_r:.0f}, {actual_g:.0f}, {actual_b:.0f})', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
